In [1]:
import json
import os
import glob
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
# ------------------------------------------------------
# Chunk text using RecursiveCharacterTextSplitter
# ------------------------------------------------------

def create_text_splitter(chunk_size=500, chunk_overlap=50):
    """Create a RecursiveCharacterTextSplitter with language-aware separators."""
    return RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " "],  # Split paragraphs first, then sentences, then words
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False
    )

def chunk_text(text, splitter):
    """Split text into chunks using RecursiveCharacterTextSplitter."""
    if not text or not text.strip():
        return []
    return splitter.split_text(text)

def chunk_pages(pages, chunk_size=500, chunk_overlap=50):
    """
    Split pages into chunks while preserving metadata.
    
    Args:
        pages: List of page dictionaries with text and metadata
        chunk_size: Maximum size of each chunk in characters
        chunk_overlap: Number of overlapping characters between chunks
    
    Returns:
        List of chunk dictionaries with chunk_id, text, and metadata
    """
    splitter = create_text_splitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = []
    chunk_counter = 0
    
    for page in pages:
        text = page.get("text", "")
        split_texts = chunk_text(text, splitter)
        
        if not split_texts:
            continue
            
        for i, chunk_text_content in enumerate(split_texts):
            chunk_counter += 1
            chunks.append({
                "chunk_id": chunk_counter,
                "page_number": page.get("page_number"),
                "text": chunk_text_content,
                "category_id": page.get("category_id"),
                "category_name": page.get("category_name"),
                "subcategory": page.get("subcategory", "")
            })
    
    print(f"Created {len(chunks)} total chunks from {len(pages)} pages.")
    return chunks

In [4]:
# ------------------------------------------------------
# Save JSON
# ------------------------------------------------------
def save_json(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved → {path}")


In [3]:
# Helper function to load JSON files
def load_json(path):
    """Load JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def process_json_files(input_files, output_file, chunk_size=500, chunk_overlap=50):
    """
    Process one or multiple JSON files and chunk their content.
    
    Args:
        input_files: List of input file paths or single file path
        output_file: Path where chunked output will be saved
        chunk_size: Size of text chunks in characters
        chunk_overlap: Overlap between chunks
    
    Returns:
        List of chunked data
    """
    if isinstance(input_files, str):
        input_files = [input_files]
    
    all_pages = []
    
    # Load all pages from input files
    for file_path in input_files:
        if os.path.exists(file_path):
            print(f"Loading: {file_path}")
            data = load_json(file_path)
            all_pages.extend(data)
            print(f"  Loaded {len(data)} pages")
    
    if not all_pages:
        print(f"⚠ No pages found in input files")
        return []
    
    # Chunk all pages
    print(f"\nChunking {len(all_pages)} pages (chunk_size={chunk_size}, overlap={chunk_overlap})...")
    chunks = chunk_pages(all_pages, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    
    # Save chunks
    if chunks:
        save_json(chunks, output_file)
    
    return chunks

In [ ]:
# Paths for 211data
INPUT_211_CATEGORIZED = "../data/processed/211data/pages_with_categories.json"
OUTPUT_211_CHUNKS = "../data/chuncked/211data/chunks_211data.json"

# Paths for macommunaute
INPUT_MACOMMUNAUTE_DIR = "../data/processed/macommunaute/"
OUTPUT_MACOMMUNAUTE_DIR = "../data/chuncked/macommunaute/"
# Chunking parameters
CHUNK_SIZE = 500       # Size of each chunk in characters
CHUNK_OVERLAP = 50     # Overlap between chunks

In [6]:
# Process 211data categorized file
print("="*60)
print("Processing 211data Categorized File")
print("="*60)

chunks_211 = process_json_files(
    INPUT_211_CATEGORIZED,
    OUTPUT_211_CHUNKS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

Processing 211data Categorized File
Loading: ../data/processed/211data/pages_with_categories.json
  Loaded 3090 pages

Chunking 3090 pages (chunk_size=500, overlap=50)...
Created 21608 total chunks from 3090 pages.
Saved → ../data/interim/211data/chunks_211data.json


In [7]:
# Process macommunaute categorized files
print(f"\n{'='*60}")
print("Processing Macommunaute Categorized Files")
print(f"{'='*60}\n")

# Find all macommunaute categorized JSON files
macommunaute_files = glob.glob(os.path.join(INPUT_MACOMMUNAUTE_DIR, "*_categorized.json"))
print(f"Found {len(macommunaute_files)} macommunaute categorized files\n")

for mac_file in sorted(macommunaute_files):
    filename = os.path.basename(mac_file)
    print(f"Processing: {filename}")
    
    # Create output filename (remove _categorized.json, add _chunks.json)
    base_name = filename.replace("_categorized.json", "")
    output_file = os.path.join(OUTPUT_MACOMMUNAUTE_DIR, f"{base_name}_chunks.json")
    
    # Process and chunk
    chunks = process_json_files(
        mac_file,
        output_file,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    print()


Processing Macommunaute Categorized Files

Found 11 macommunaute categorized files

Processing: aide_ecoute_en_categorized.json
Loading: ../data/processed/macommunaute\aide_ecoute_en_categorized.json
  Loaded 32 pages

Chunking 32 pages (chunk_size=500, overlap=50)...
Created 136 total chunks from 32 pages.
Saved → ../data/interim/macommunaute/aide_ecoute_en_chunks.json

Processing: alimentation_en_categorized.json
Loading: ../data/processed/macommunaute\alimentation_en_categorized.json
  Loaded 7 pages

Chunking 7 pages (chunk_size=500, overlap=50)...
Created 33 total chunks from 7 pages.
Saved → ../data/interim/macommunaute/alimentation_en_chunks.json

Processing: communautes_culturelles_nouveaux_arrivants_en_categorized.json
Loading: ../data/processed/macommunaute\communautes_culturelles_nouveaux_arrivants_en_categorized.json
  Loaded 40 pages

Chunking 40 pages (chunk_size=500, overlap=50)...
Created 159 total chunks from 40 pages.
Saved → ../data/interim/macommunaute/communautes_

In [8]:
# Verification: Show summary of chunked files
print(f"\n{'='*60}")
print("Verification Summary")
print(f"{'='*60}\n")

# Summary for 211data chunks
if os.path.exists(OUTPUT_211_CHUNKS):
    data_211 = load_json(OUTPUT_211_CHUNKS)
    print(f"✓ 211data Chunks: {OUTPUT_211_CHUNKS}")
    print(f"  Total chunks: {len(data_211)}")
    if data_211:
        sample = data_211[0]
        print(f"  Sample chunk:")
        print(f"    ID: {sample['chunk_id']}")
        print(f"    Page: {sample['page_number']}")
        print(f"    Category: {sample['category_name']}")
        print(f"    Text length: {len(sample['text'])} chars")
        print(f"    Text preview: {sample['text'][:100]}...\n")

# Summary for macommunaute chunks
output_files = glob.glob(os.path.join(OUTPUT_MACOMMUNAUTE_DIR, "*_chunks.json"))
if output_files:
    print(f"✓ Macommunaute Chunks: {len(output_files)} files")
    for output_file in sorted(output_files):
        data = load_json(output_file)
        filename = os.path.basename(output_file)
        print(f"  {filename}: {len(data)} chunks")
        if data:
            sample = data[0]
            print(f"    Category: {sample['category_name']}, Text length: {len(sample['text'])} chars")


Verification Summary

✓ 211data Chunks: ../data/interim/211data/chunks_211data.json
  Total chunks: 21608
  Sample chunk:
    ID: 1
    Page: 1
    Category: Sports, Recreation and Culture
    Text length: 101 chars
    Text preview: DIRECTORY OF
SOCIAL AND COMMUNITY
RESOURCES
MONTRÉAL / MONTRÉAL / CÔTE-DES-NEIGES—NOTRE-DAME-DE-GRÂC...

✓ Macommunaute Chunks: 11 files
  aide_ecoute_en_chunks.json: 136 chunks
    Category: Mental Health and addictions, Text length: 476 chars
  alimentation_en_chunks.json: 33 chunks
    Category: Food, Text length: 478 chars
  communautes_culturelles_nouveaux_arrivants_en_chunks.json: 159 chunks
    Category: Immigration and cultural communities, Text length: 424 chars
  education_en_chunks.json: 138 chunks
    Category: Education and Literacy, Text length: 432 chars
  emploi_en_chunks.json: 33 chunks
    Category: Employment and Income Support, Text length: 481 chars
  famille_en_chunks.json: 237 chunks
    Category: Child, Youth and Family, Text lengt

In [ ]:
print(f"\n{'='*60}")
print("✓ All files processed successfully!")
print(f"{'='*60}")

Saved → ../data/processed/pages_with_categories.json
